# Prototype replication (EXACT grid): new `PatternSearchCV`, 195-point space

Correction of `Prototype_Replication.ipynb`, which used a 26-value
`n_estimators` axis; the recorded prototype run actually used
`[10, 70, 130, 190, 250]`. This notebook uses the exact grid.

Replicates the exact data pipeline and search configuration of
`DatasetSize_and_ParamOpt_WORKING_(3Large_Aug_30_2025).ipynb`, replacing the
prototype `PatternSearchCV` (cell 108/110) with the production package.

**Reference results recorded in the prototype notebook** (ExtraTrees run):
- Prototype pattern search: `Total Fits: 10` exploration fits **+ 1 initial = 11 evaluations**,
  all on 100% of the data; best MAE **805.038** at `{max_features: 4, n_estimators: 130, max_depth: 17}`; 1212.28 s (on the original machine).
- skopt Bayesian optimization: `gp_minimize(..., n_calls=15, random_state=0)` = **15 evaluations**
  (skopt enforces a minimum of 10), all on 100% of the data. Note: its objective
  maximized R2, not MAE, so only the *fit count* is compared, not the score.

In [1]:
# --- Data pipeline: byte-for-byte replication of the prototype notebook ---
import time
import numpy as np
import pandas as pd

t0 = time.time()
trainBench = pd.read_csv(r"C:\FILES\Code\Benchmarking\train.csv")  # notebook: /home/dell/train.csv

SplitPoint = int(len(trainBench.index) * 0.8)
print("SplitPoint: ", SplitPoint)

df = trainBench

# int64 -> int32; object -> category -> numeric codes (Date included, as in the prototype)
Int64columns = df.select_dtypes(["int64"]).columns
df[Int64columns] = df[Int64columns].astype(np.int32)
cat_columns = df.select_dtypes(["object"]).columns
df[cat_columns] = df[cat_columns].astype("category")
df[cat_columns] = df[cat_columns].apply(lambda x: x.cat.codes)

# the five weather columns the prototype drops (note the dataset's own 'kM' typo)
df = df.drop("Max_Gust_SpeedKm_h", axis=1)
df = df.drop("CloudCover", axis=1)
df = df.drop("Max_VisibilityKm", axis=1)
df = df.drop("Min_VisibilitykM", axis=1)
df = df.drop("Mean_VisibilityKm", axis=1)

# positional 80/20 chronological split
trainBench, validBench = df.iloc[:SplitPoint, :], df.iloc[SplitPoint:, :]
print("Training Set shape", trainBench.shape)
print("Validation Set shape", validBench.shape)
del df

# feature mask: everything but the target (alphabetical order, incl. Date codes
# and NumberOfCustomers, exactly as the prototype had it)
mask = trainBench.columns.difference(["NumberOfSales"])
trainDataset_X = trainBench[mask]
trainDataset_y = trainBench["NumberOfSales"]
validBench_X = validBench[mask]
validBench_y = validBench["NumberOfSales"]
print("Feature Columns:")
print(list(mask))
del trainBench, validBench
print(f"Pipeline done in {time.time() - t0:.1f} s")

SplitPoint:  418416


C:\Users\Home\AppData\Local\Temp\ipykernel_16536\4255920482.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_columns = df.select_dtypes(["object"]).columns


Training Set shape (418416, 31)
Validation Set shape (104605, 31)
Feature Columns:
['AssortmentType', 'Date', 'Events', 'HasPromotions', 'IsHoliday', 'IsOpen', 'Max_Dew_PointC', 'Max_Humidity', 'Max_Sea_Level_PressurehPa', 'Max_TemperatureC', 'Max_Wind_SpeedKm_h', 'Mean_Dew_PointC', 'Mean_Humidity', 'Mean_Sea_Level_PressurehPa', 'Mean_TemperatureC', 'Mean_Wind_SpeedKm_h', 'Min_Dew_PointC', 'Min_Humidity', 'Min_Sea_Level_PressurehPa', 'Min_TemperatureC', 'NearestCompetitor', 'NumberOfCustomers', 'Precipitationmm', 'Region', 'Region_AreaKM2', 'Region_GDP', 'Region_PopulationK', 'StoreID', 'StoreType', 'WindDirDegrees']
Pipeline done in 6.1 s


In [2]:
# --- Model, grid and CV: identical to the prototype execution block ---
import os
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import TimeSeriesSplit

X, y = trainDataset_X, trainDataset_y

N_features = X.shape[1]
clf = ExtraTreesRegressor(n_estimators=240,
                          max_features=max(1, N_features - 15),
                          max_depth=16,
                          n_jobs=1,
                          random_state=0)

param_grid = {
    "n_estimators": [10, 70, 130, 190, 250],  # EXACT grid of the recorded
    # prototype run (np.linspace(10, 250, 5)); the 26-value list in earlier
    # notebook cells was commented out in the cell that actually ran
    "max_features": [2, 3, 4],
    "max_depth": [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17],
}

tscv = TimeSeriesSplit(n_splits=5)
print("cores available:", os.cpu_count())

cores available: 8


In [3]:
# --- The new PatternSearchCV, default configuration (bullseye ON) ---
from pattern_search_cv import PatternSearchCV

search = PatternSearchCV(
    clf,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=tscv,
    n_jobs=-1,
    random_state=0,
    verbose=1,
)

start = time.time()
search.fit(X, y)
elapsed = time.time() - start
print(f"\nPatternSearchCV took {elapsed:.2f} seconds.")

[pattern_search_cv] subsample mode=expanding, zones=[0.1, 0.2, 0.5, 1.0] (rows [41842, 83684, 209208, 418416])


[pattern_search_cv] starts (1): [{'n_estimators': 130, 'max_features': 3, 'max_depth': 11}]


[pattern_search_cv] poll='auto' resolved to 'opportunistic' (n_jobs=-1, n_splits=5)


Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


...

Fitting 5 folds for each of 1 candidates, totalling 5 fits


Fitting 5 folds for each of 1 candidates, totalling 5 fits


[pattern_search_cv] climber 0 converged at {'n_estimators': 130, 'max_features': 4, 'max_depth': 17} score=-805.038060573306


[pattern_search_cv] engine done: 17 evaluations, 16 cache hits, climbers: {0: 'converged'}



PatternSearchCV took 684.61 seconds.


In [4]:
# --- Results and the comparison the prototype notebook made ---
res = search.cv_results_
n_fits = len(res["params"])
n_rows = len(y)
weighted = float(np.sum(np.asarray(res["n_resources"]) / n_rows))

print("=" * 64)
print("NEW PatternSearchCV (defaults: 1 start, zones 10/20/50/100)")
print("=" * 64)
print(f"evaluations           : {n_fits}")
print(f"full-fit equivalents  : {weighted:.2f}  (fits weighted by data fraction)")
print(f"cache hits            : {search.n_cache_hits_}")
print(f"wall-clock            : {elapsed:.1f} s")
print(f"best params           : {search.best_params_}")
print(f"best CV MAE           : {-search.best_score_:.3f}")

from sklearn.metrics import mean_absolute_error
val_mae = mean_absolute_error(validBench_y, search.predict(validBench_X))
print(f"held-out (20%) MAE    : {val_mae:.3f}")

print()
print("=" * 64)
print("COMPARISON (references recorded in the prototype notebook)")
print("=" * 64)
proto_evals, proto_mae = 11, 805.038
bayes_evals = 15
print(f"{'method':34s}{'evals':>7s}{'full-fit equiv':>16s}")
print(f"{'prototype pattern search':34s}{proto_evals:>7d}{proto_evals:>16.2f}")
print(f"{'skopt gp_minimize (n_calls=15)':34s}{bayes_evals:>7d}{bayes_evals:>16.2f}")
print(f"{'NEW PatternSearchCV (defaults)':34s}{n_fits:>7d}{weighted:>16.2f}")
print()
print(f"prototype best CV MAE : {proto_mae:.3f}")
print(f"new best CV MAE       : {-search.best_score_:.3f}")
print()
print("distinct local optima found:")
for o in search.local_optima_:
    print("  ", o["params"], "CV MAE", f"{-o['score']:.3f}")

NEW PatternSearchCV (defaults: 1 start, zones 10/20/50/100)
evaluations           : 17
full-fit equivalents  : 6.20  (fits weighted by data fraction)
cache hits            : 16
wall-clock            : 684.6 s
best params           : {'n_estimators': 130, 'max_features': 4, 'max_depth': 17}
best CV MAE           : 805.038


held-out (20%) MAE    : 780.184

COMPARISON (references recorded in the prototype notebook)
method                              evals  full-fit equiv
prototype pattern search               11           11.00
skopt gp_minimize (n_calls=15)         15           15.00
NEW PatternSearchCV (defaults)         17            6.20

prototype best CV MAE : 805.038
new best CV MAE       : 805.038

distinct local optima found:
   {'n_estimators': 130, 'max_features': 4, 'max_depth': 17} CV MAE 805.038


**Reading the numbers**

- *evaluations*: model configurations scored (each = 5 TimeSeriesSplit fits).
  The prototype and gp_minimize always paid full price (100% of rows); the new
  version pays by the zone the climber was in, so *full-fit equivalents* is the
  honest cost metric: an evaluation at 10% data counts as 0.1.
- CV MAE comparability caveat: the prototype scored every candidate on 100% of
  the data; the new version's `best_score_` is also a full-data score (best_*
  only come from fraction-1.0 rows), so the two best-MAE numbers ARE comparable.
- gp_minimize optimized R2 rather than MAE in the prototype notebook, so only
  its fit count is compared, not its score. skopt is archived and cannot run
  on this numpy 2.x stack; its numbers are quoted from the notebook record.